# Mi proyecto · De la teoría a la práctica (Sesiones 1 a 3)
**Plantilla espejo del caso guiado Olist** · Módulo: Preparación, Recopilación y Análisis Exploratorio de Datos · MIADAS – UMSA

**Grupo:1**

*   Denisse Marin Ayala
*   Dardo Gamarra
*   Nelson De La Quintana
*   Mauricio De La Quintana

**Conjunto de datos:** The Movies Dataset · **Fuente y URL:** Kaggle (`https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset`)

Esta plantilla tiene **las mismas secciones** que el caso guiado. Repite cada paso con **tu conjunto de datos**. Este notebook evolucionará en las próximas sesiones con la limpieza, el análisis exploratorio, la normalización y la calidad.

### Requisitos mínimos de tu conjunto de datos
- [X] Al menos **1 000 registros**
- [X] Al menos **8 variables** que mezclen escalas (nominal, ordinal, de razón)
- [X] Al menos **una variable de fecha**
- [X] **Problemas de calidad** reales (faltantes, duplicados o inconsistencias)
- [X] Idealmente, **dos tablas o fuentes** que se puedan integrar
- [X] **Licencia** que permita su uso académico

### Paso 0 · Cargar los Datos  · (Kaggle)
Kaggle no permite leer el archivo por su dirección. Usa la librería oficial **kagglehub**: descarga los datasets **públicos** sin iniciar sesión.

1. Copia el identificador de la dirección del dataset: en `kaggle.com/datasets/olistbr/brazilian-ecommerce` es `olistbr/brazilian-ecommerce`.
2. Quita el `#` de las líneas de la celda, reemplaza el identificador y el nombre del archivo, y ejecuta.

> Si el dataset pide **aceptar reglas** (por ejemplo, los de competencias), necesitarás un token de Kaggle: mejor elige otro dataset.

In [16]:
# !pip install kagglehub        # solo si Colab indica que no está instalada
# import os, kagglehub
# ruta = kagglehub.dataset_download("usuario/nombre-del-dataset")
# print(os.listdir(ruta))                                  # muestra los archivos descargados
# datos = pd.read_csv(os.path.join(ruta, "nombre_del_archivo.csv"))
# print(datos.shape)
import math
import os
import kagglehub
import pandas as pd

path = kagglehub.dataset_download("rounakbanik/the-movies-dataset")

movies   = pd.read_csv(os.path.join(path, "movies_metadata.csv"), low_memory=False)
credits  = pd.read_csv(os.path.join(path, "credits.csv"), low_memory=False)
keywords = pd.read_csv(os.path.join(path, "keywords.csv"), low_memory=False)
links    = pd.read_csv(os.path.join(path, "links.csv"), low_memory=False)

print("Películas: ", movies.shape)
print("Créditos:  ", credits.shape)
print("Palabras:  ", keywords.shape)
print("Enlaces:   ", links.shape)

Using Colab cache for faster access to the 'the-movies-dataset' dataset.
Películas:  (45466, 24)
Créditos:   (45476, 3)
Palabras:   (46419, 2)
Enlaces:    (45843, 3)


---
# S1 · Tipos de datos y escalas (Sesión 1)
### S1.1 · Tipos de dato
¿Qué columnas se leyeron con un tipo incorrecto (fechas como texto, códigos como números)?

In [17]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  object 
 1   belongs_to_collection  4494 non-null   object 
 2   budget                 45466 non-null  object 
 3   genres                 45466 non-null  object 
 4   homepage               7782 non-null   object 
 5   id                     45466 non-null  object 
 6   imdb_id                45449 non-null  object 
 7   original_language      45455 non-null  object 
 8   original_title         45466 non-null  object 
 9   overview               44512 non-null  object 
 10  popularity             45461 non-null  object 
 11  poster_path            45080 non-null  object 
 12  production_companies   45463 non-null  object 
 13  production_countries   45463 non-null  object 
 14  release_date           45379 non-null  object 
 15  re

### S1.2 · Corrige los tipos
Convierte las fechas con `pd.to_datetime` y los códigos (identificadores, códigos postales) a texto con `astype(str)`.

In [18]:
# datos["columna_fecha"] = pd.to_datetime(datos["columna_fecha"])
# datos["columna_codigo"] = datos["columna_codigo"].astype(str)

# Convertimos la fecha de estreno a datetime
movies["release_date"] = pd.to_datetime(movies["release_date"], errors="coerce")

# Aseguramos que el identificador 'id' sea texto
movies["id"] = movies["id"].astype(str)

print("\nFechas convertidas con éxito:")
print(movies[["release_date"]].dtypes)
print("Primer lanzamiento:", movies["release_date"].min())
print("Último lanzamiento:", movies["release_date"].max())


Fechas convertidas con éxito:
release_date    datetime64[ns]
dtype: object
Primer lanzamiento: 1874-12-09 00:00:00
Último lanzamiento: 2020-12-16 00:00:00


In [19]:
# S1.2 · Corrección de tipos de datos adicionales
movies["release_date"] = pd.to_datetime(movies["release_date"], errors="coerce")
movies["id"] = movies["id"].astype(str)

# Limpiamos y convertimos variables financieras y de razón a numérico
movies["budget"] = pd.to_numeric(movies["budget"], errors="coerce")
movies["popularity"] = pd.to_numeric(movies["popularity"], errors="coerce")

print("Tipos de datos corregidos y limpios.")

Tipos de datos corregidos y limpios.


###**Observaciones en los tipos de datos:**
Al revisar movies.info(), detectamos 3 problemas típicos de datos sin tratar:


---
> 1. **Fechas como texto:** `release_date` se leyó como objeto y requirió conversión a `datetime`.
> 2. **Variables financieras corruptas:** `budget` se leía como texto debido a anomalías de formato en filas específicas, por lo que fue necesario forzar su conversión a tipo numérico.
> 3. **Identificadores:** `id` se mantiene estrictamente como texto (`str`) por ser una variable nominal.



### S1.3 · Escala de cada variable
Completa la tabla con tus variables (mínimo 8):

| Variable | Escala (nominal / ordinal / intervalo / razón) | Cálculos válidos |
|---|---|---|
|`original_language`(Lenguaje de Origen)|Nominal|Frecuencias, moda|
|`genres`(Generos)|Nominal|Frecuencias, proporciones|
|`vote_average`(Promedio de Votos)|Ordinal/Intervalo|Mediana, percentiles, media|
|`release_date`(Fecha de Estreno)|Intervalo|Diferencias entre fechas, rangos|
|`budget`, `revenue`(presupuesto)|Razón|Todos, incluidas razones y proporciones|
|`runtime(Duración)`|Razón|Media, mediana, desviación estándar|
|`popularity`(Popularidad)|Razón|Media, mediana, percentiles|
|`id`(identificador)|Nominal|Conteo de registros únicos|

In [20]:
# Frecuencias de una variable nominal u ordinal:
# datos["columna"].value_counts()
print("--- Frecuencias de idioma original ---")
print(movies["original_language"].value_counts().head(5))

# Estadísticos de variables de razón
# datos[["columna1", "columna2"]].describe().round(2)
print("\n--- Estadísticos de variables de razón ---")
movies[["budget", "revenue", "runtime", "vote_average"]].describe().round(2)

--- Frecuencias de idioma original ---
original_language
en    32269
fr     2438
it     1529
ja     1350
de     1080
Name: count, dtype: int64

--- Estadísticos de variables de razón ---


,budget,revenue,runtime,vote_average
count,4.546300e+04,4.546000e+04,45203.00,45460.00
mean,4.224579e+06,1.120935e+07,94.13,5.62
std,1.742413e+07,6.433225e+07,38.41,1.92
min,0.000000e+00,0.000000e+00,0.00,0.00
25%,0.000000e+00,0.000000e+00,85.00,5.00
50%,0.000000e+00,0.000000e+00,95.00,6.00
75%,0.000000e+00,0.000000e+00,107.00,6.80
max,3.800000e+08,2.787965e+09,1256.00,10.00


---
# S2 · Muestreo (Sesión 2)
¿Qué subconjunto necesitarías revisar a mano o encuestar? Calcula el tamaño de muestra y compara una muestra aleatoria con una estratificada.

In [21]:
def tamano_muestra(N, e=0.05, z=1.96, p=0.5):
    n0 = z**2 * p * (1 - p) / e**2
    return math.ceil(n0 / (1 + (n0 - 1) / N))
# Cochran con el total de filas de la tabla movies
n = tamano_muestra(len(movies))
print(f"Poblacion Total de peliculas (N): {len(movies)} | Tamaño de muestra (n): {n}")


Poblacion Total de peliculas (N): 45466 | Tamaño de muestra (n): 381


In [22]:
# variable_estrato = "columna_categorica"
# aleatoria = datos.sample(n=n, random_state=42)
# estratificada = datos.groupby(variable_estrato, group_keys=False).sample(frac=n / len(datos), random_state=42)
# pd.DataFrame({
#     "Población": datos[variable_estrato].value_counts(normalize=True),
#     "Aleatoria": aleatoria[variable_estrato].value_counts(normalize=True),
#     "Estratificada": estratificada[variable_estrato].value_counts(normalize=True),
# }).round(3)


# Muestreo estratificado usando el idioma original como estrato
variable_estrato = "original_language"
movies_filtradas = movies.dropna(subset=[variable_estrato]).copy()
top_idiomas = movies_filtradas[variable_estrato].value_counts().head(5).index
movies_filtradas = movies_filtradas[movies_filtradas[variable_estrato].isin(top_idiomas)]
fraccion = n / len(movies_filtradas)

aleatoria = movies_filtradas.sample(n=n, random_state=42)
estratificada = movies_filtradas.groupby(variable_estrato, group_keys=False).sample(frac=fraccion, random_state=42)

comparacion = pd.DataFrame({
    "Población": movies_filtradas[variable_estrato].value_counts(normalize=True),
    "Aleatoria": aleatoria[variable_estrato].value_counts(normalize=True),
    "Estratificada": estratificada[variable_estrato].value_counts(normalize=True),
}).sort_index().round(3)

comparacion

,Población,Aleatoria,Estratificada
original_language,,,
de,0.028,0.045,0.029
en,0.835,0.780,0.835
fr,0.063,0.084,0.063
it,0.040,0.050,0.039
ja,0.035,0.042,0.034


**Aplicación práctica del muestreo y conclusión:**
> Supongamos que una distribuidora quiere auditar manualmente la calidad de las sinopsis del catálogo. Revisar las **45 466 películas** a mano es inviable, por lo que calculamos una muestra representativa con Cochran (**n = 381** al 95 % de confianza y 5 % de error).
> Al comparar las muestras, se observa que el muestreo estratificado reproduce de forma exacta la proporción real de la población en los idiomas principales, mientras que el muestreo aleatorio simple tiende a fluctuar más, validando la utilidad de estratificar cuando hay categorías dominantes.

---
# S3 · Fuentes e integración (Sesión 3)
Si tu conjunto tiene dos o más tablas, únelas por su clave. Si tiene una sola, identifica qué fuente externa podría complementarla (por ejemplo, datos del INE).

In [23]:
# otra_tabla = pd.read_csv("dirección_de_la_otra_tabla.csv")
# integrado = datos.merge(otra_tabla, on="clave_comun", how="left")
# print(integrado.shape)

# Asegurmos que la clave 'id' sea texto en ambas tblas
credits["id"] = credits["id"].astype(str)

# Integración
peliculas_integradas = movies.merge(credits, on="id", how="left")

print("Dimensiones de la tabla integrada resultante:", peliculas_integradas.shape)
peliculas_integradas[["title", "cast", "crew"]].head(3)

Dimensiones de la tabla integrada resultante: (45542, 26)


,title,cast,crew
0,Toy Story,"[{'cast_id': 14, 'character': 'Woody (voice)',...","[{'credit_id': '52fe4284c3a36847f8024f49', 'de..."
1,Jumanji,"[{'cast_id': 1, 'character': 'Alan Parrish', '...","[{'credit_id': '52fe44bfc3a36847f80a7cd1', 'de..."
2,Grumpier Old Men,"[{'cast_id': 2, 'character': 'Max Goldman', 'c...","[{'credit_id': '52fe466a9251416c75077a89', 'de..."


---
# Cierre · Informe de recolección inicial (CRISP-DM, fase 2)

| Elemento | Mi conjunto de datos |
|---|---|
| **Fuente** |Kaggle · *The Movies Dataset* (recopilado de metadatos de TMDb y GroupLens)|
| **Tipo de fuente** (interna / externa · primaria / secundaria) |Externa · datos secundarios extraídos de registros historicos de la industria cinematográfica|
| **Formato y acceso** (CSV, Excel, API… · URL) |Múltiples archivos CSV (movies_metadata, credits, keywords, links), descargados de forma programática con kagglehub en Colab|
| **Tablas y número de registros** |movies (45.466 registros), credits, keywords y links|
| **Periodo que cubre** |Registros históricos acumulados de la industria de cine hasta la fecha de actualización de la fuente|
| **Problemas encontrados** |Fechas leídas como texto (`object`), valores nulos o ceros en variables financieras (`budget`, `revenue`), y campos de listas estructuradas en texto plano en los créditos|
| **Primer hallazgo o pregunta de negocio** |¿Qué relación existe entre el presupuesto invertido, el género y la duración de una película con su recaudación en taquilla y la calificación promedio otorgada por el público?|